In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## fmri text

In [5]:
fmri = pd.read_csv('../../data/results/supplement/fmri_text.csv')
fmri = fmri.pivot(index='embed', columns='norm', values='r2_mean')
fmri

norm,aoa_glasgow,arousal_glasgow,concreteness_glasgow,dominance_glasgow,familiarity_glasgow,gender_association_glasgow,imageability_glasgow,semantic_size_glasgow,valence_glasgow
embed,,,,,,,,,
fMRI_text_cognival,-0.019051,-0.044966,-0.016066,-0.030578,-0.020753,-0.023693,-0.020584,-0.038038,-0.045542
fMRI_text_denoise_1024d,-0.019867,-0.030493,-0.023564,-0.033547,-0.042031,-0.053244,-0.026823,-0.026573,-0.054900
fMRI_text_denoise_128d,-0.020800,-0.044686,0.016987,-0.035331,-0.030157,-0.026060,0.025236,-0.009469,-0.054036
fMRI_text_denoise_256d,-0.019359,-0.044300,-0.013855,-0.030513,-0.029341,-0.026146,-0.015677,-0.038761,-0.050247
fMRI_text_denoise_512d,-0.019178,-0.030602,0.007363,-0.040585,-0.029808,-0.026336,0.028134,-0.038750,-0.052029


In [6]:
winner_mask = fmri.apply(lambda col: col == col.max(), axis=0)
winner_mask

norm,aoa_glasgow,arousal_glasgow,concreteness_glasgow,dominance_glasgow,familiarity_glasgow,gender_association_glasgow,imageability_glasgow,semantic_size_glasgow,valence_glasgow
embed,,,,,,,,,
fMRI_text_cognival,True,False,False,False,True,True,False,False,True
fMRI_text_denoise_1024d,False,True,False,False,False,False,False,False,False
fMRI_text_denoise_128d,False,False,True,False,False,False,False,True,False
fMRI_text_denoise_256d,False,False,False,True,False,False,False,False,False
fMRI_text_denoise_512d,False,False,False,False,False,False,True,False,False


In [ ]:
def annotate(df, ax):
    for x, norm_cat in enumerate(df.columns):
        for y, embed in enumerate(df.index):
            annot = df.loc[embed, norm_cat]

            # Scientific notation
            if abs(annot) > 1e3:
                annot = f'{annot:.1e}'
            elif np.isnan(annot):
                annot = ''
            else:
                annot = f'{annot:.2f}'

            # Fontsize and fontweight
            if winner_mask.loc[embed, norm_cat]:
                fontsize, fontweight = 15, 'bold'
            else:
                fontsize, fontweight = 11, 'normal'


            ax.text(
                x + .5, y + .6, annot, fontsize=fontsize, fontweight=fontweight,
                ha='center', va='center', color='black'
            )

top_behav = (
    rca_avg_piv[[embed_to_type[embed] == 'behavior' for embed in rca_avg_piv.index]] # Selects behavior embeds
    .mean(axis=1).idxmax() # Selects the behavior embed with the highest average r2
)

# Sorts norms by the average r2 of the top behavior embed
norm_ord = rca_avg_piv.loc[top_behav].sort_values(ascending=True).index

# Builds heatmap dfs
heat_dfs = {}
embed_types = ['text', 'brain', 'behavior']
for embed_type in embed_types:
    heat_df = rca_avg_piv[[embed_to_type[embed] == embed_type for embed in rca_avg_piv.index]]

    # Sorts index and columns
    embed_order = heat_df.mean(axis=1).sort_values(ascending=False).index
    heat_dfs[embed_type] = heat_df[norm_ord].loc[embed_order]

# renaming funct
def fix_names(name: pd.Series) -> pd.Series:
    rename = {
        'SVD_sim_rel': 'SVD_similarity_relatedness',
        'SGSoftMaxInput_SWOW': 'SkipGramInput SWOW',
        'SGSoftMaxOutput_SWOW': 'SkipGramOutput SWOW'
    }
    return rename.get(name, name).replace('_', ' ')

fig, axs = plt.subplots(3, 1, figsize=(18, 10))

vmax = rca_avg_piv.max().max()
for i, embed_type in enumerate(['text', 'behavior', 'brain']):
    heat_df = heat_dfs[embed_type]

    sns.heatmap(
        heat_df, ax=axs[i], vmin=0, cmap=lighter_viridis,
        vmax=vmax, annot=False, fmt='', cbar=False,

    )


    axs[i].set(xlabel='', xticklabels=[])

    # sets ylabel on right-hand side and flips it
    axs[i].set_ylabel(
        embed_type.title(), fontsize=17, rotation=270,
        labelpad=20, va='center', ha='center'
    )
    axs[i].yaxis.set_label_position('right')

    # Annotates cells
    annotate(heat_df, axs[i])


    # Ensure y-axis labels match the number of ticks
    axs[i].set_yticks(pd.Series(range(len(heat_df.index))) + .5)
    heat_df.index = heat_df.index.to_series().apply(fix_names)
    axs[i].set_yticklabels(heat_df.index, fontsize=12)

# Adding xticklabels to last plot
norm_ord = norm_ord.str.title().str.replace(' Of ', ' of ', regex=True)
axs[-1].set_xticklabels(norm_ord, rotation=90, fontsize=13)

# Sets figure title
axs[0].set_title('Average Test ${R^2}$', fontsize=20)